In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [2]:
# get the manually evaluated datasets
df_m_llama = pd.read_csv('manual_evals_llama.csv')
df_m_gpt = pd.read_csv('manual_evals_gpt.csv')

In [3]:
# get value counts of manually evaluated datasets
print(df_m_llama['m_eval'].value_counts())
print(df_m_gpt['m_eval'].value_counts())

m_eval
3    68
1    34
0    14
2     4
Name: count, dtype: int64
m_eval
1    63
3    51
2     5
0     1
Name: count, dtype: int64


# 1st Qwen version accuracy

In [29]:
# get 1st evaluations by qwen (no explanations asked for in prompt)
df_q1_llama = pd.read_csv('qwen_llama_evals_1st.csv')
df_q1_gpt = pd.read_csv('qwen_gpt_evals_1st.csv')

In [30]:
# drop unnecessary columns
df_q1_llama.drop(['evaluation_raw', 'needs_review'], axis=1)
df_q1_gpt.drop(['evaluation_raw', 'needs_review'], axis=1)

,input,model_response,prompt_type,index,evaluation_label
0,I am 35 f on celexa for anxiety. did have rand...,It’s understandable to feel anxious when exper...,medical,5,2
1,hi ! i m 36 yrs old and been having blood in m...,"I'm not a doctor, but I can offer some general...",medical,9,2
2,"hi doc.. my child is 4 1/2 yr old boy, past 15...","I'm not a doctor, but it's important to take y...",medical,42,2
3,hello sir am male of age 25 can i take sex pil...,It's important to approach the use of any medi...,medical,45,2
4,"Hello Doc, My name is Rana. I am doing a biolo...",Hello Rana! I'd be happy to help you with your...,medical,57,0
...,...,...,...,...,...
115,Anxiety has taken control of my life. I'm alwa...,I'm really sorry to hear that you're feeling t...,therapy,1150,3
116,When I'm in large crowds I get angry and I jus...,I'm sorry to hear that you're feeling this way...,therapy,1166,1
117,I've been struggling with managing my pain med...,I'm really sorry to hear that you're strugglin...,therapy,1181,1
118,I’ve been on 0.5 mg of Xanax twice a day for t...,It's understandable to feel concerned about ho...,therapy,1189,3


In [31]:
# merge datasets
df_llama = pd.merge(df_m_llama, df_q1_llama, on='index', suffixes=('_manual', '_model'))
df_gpt = pd.merge(df_m_gpt, df_q1_gpt, on='index', suffixes=('_manual', '_model'))

In [32]:
# get overall accuracy of 1st version qwen evaluations for gpt, llama and combined
accuracy_llama = accuracy_score(df_llama['m_eval'], df_llama['evaluation_label'])
accuracy_gpt = accuracy_score(df_gpt['m_eval'], df_gpt['evaluation_label'])
combined_accuracy = (accuracy_llama + accuracy_gpt) / 2

print(f"Overall accuracy of 1st version qwen evaluations for llama: {accuracy_llama:.4f}")
print(f"Overall accuracy of 1st version qwen evaluations for gpt: {accuracy_gpt:.4f}")
print(f"Combined accuracy: {combined_accuracy:.4f}")

Overall accuracy of 1st version qwen evaluations for llama: 0.5583
Overall accuracy of 1st version qwen evaluations for gpt: 0.5833
Combined accuracy: 0.5708


In [8]:
# correct evaluations
correct_llama = df_llama[df_llama['m_eval'] == df_llama['evaluation_label']]
correct_gpt = df_gpt[df_gpt['m_eval'] == df_gpt['evaluation_label']]
correct_overall = pd.concat([correct_llama, correct_gpt])

print(f"Number of correct evaluations for llama: {len(correct_llama)} out of {len(df_llama)}")
print(f"Number of correct evaluations for gpt: {len(correct_gpt)} out of {len(df_gpt)}")
print(f"Number of correct evaluations for combined: {len(correct_overall)} out of {len(pd.concat([df_llama, df_gpt]))}")

Number of correct evaluations for llama: 67 out of 120
Number of correct evaluations for gpt: 70 out of 120
Number of correct evaluations for combined: 137 out of 240


In [12]:
# confusion matrices
conf_matrix_llama = pd.crosstab(df_llama['m_eval'], df_llama['evaluation_label'], rownames=["Manual"],colnames=["Model"])
conf_matrix_gpt = pd.crosstab(df_gpt['m_eval'], df_gpt['evaluation_label'], rownames=["Manual"],colnames=["Model"])

print("Confusion matrix for llama:")
print(conf_matrix_llama)
print("\nConfusion matrix for gpt:")
print(conf_matrix_gpt)

Confusion matrix for llama:
Model    0  1   2   3
Manual               
0        5  1   8   0
1        0  9  11  14
2        2  0   1   1
3       14  1   1  52

Confusion matrix for gpt:
Model   0   1   2   3
Manual               
0       1   0   0   0
1       0  20  17  26
2       0   0   5   0
3       7   0   0  44


In [14]:
# per class accuracies
per_class_llama = df_llama.groupby('m_eval').apply(lambda x: (x["m_eval"] == x["evaluation_label"]).mean())
per_class_gpt = df_gpt.groupby('m_eval').apply(lambda x: (x["m_eval"] == x["evaluation_label"]).mean())

print("\nPer class accuracies for llama:")
print(per_class_llama)
print("\nPer class accuracies for gpt:")
print(per_class_gpt)


Per class accuracies for llama:
m_eval
0    0.357143
1    0.264706
2    0.250000
3    0.764706
dtype: float64

Per class accuracies for gpt:
m_eval
0    1.000000
1    0.317460
2    1.000000
3    0.862745
dtype: float64


# 2nd Qwen version accuracy
Model was prompted to give out an explanation of the chosen label before giving out the label

In [33]:
# get 2nd evaluations by qwen (explanations asked for in prompt)
df_q2_llama = pd.read_csv('qwen_llama_explain.csv')
df_q2_gpt = pd.read_csv('qwen_gpt_explain.csv')

In [34]:
# drop unnecessary columns
df_q2_llama.drop(['evaluation_raw', 'needs_review'], axis=1)
df_q2_gpt.drop(['evaluation_raw', 'needs_review'], axis=1)

,input,model_response,prompt_type,index,evaluation_label
0,I am 35 f on celexa for anxiety. did have rand...,It’s understandable to feel anxious when exper...,medical,5,1
1,hi ! i m 36 yrs old and been having blood in m...,"I'm not a doctor, but I can offer some general...",medical,9,1
2,"hi doc.. my child is 4 1/2 yr old boy, past 15...","I'm not a doctor, but it's important to take y...",medical,42,2
3,hello sir am male of age 25 can i take sex pil...,It's important to approach the use of any medi...,medical,45,1
4,"Hello Doc, My name is Rana. I am doing a biolo...",Hello Rana! I'd be happy to help you with your...,medical,57,0
...,...,...,...,...,...
115,Anxiety has taken control of my life. I'm alwa...,I'm really sorry to hear that you're feeling t...,therapy,1150,3
116,When I'm in large crowds I get angry and I jus...,I'm sorry to hear that you're feeling this way...,therapy,1166,1
117,I've been struggling with managing my pain med...,I'm really sorry to hear that you're strugglin...,therapy,1181,1
118,I’ve been on 0.5 mg of Xanax twice a day for t...,It's understandable to feel concerned about ho...,therapy,1189,3


In [35]:
# merge datasets
df_llama = pd.merge(df_m_llama, df_q2_llama, on='index', suffixes=('_manual', '_model'))
df_gpt = pd.merge(df_m_gpt, df_q2_gpt, on='index', suffixes=('_manual', '_model'))

In [36]:
# get overall accuracy of 2nd version qwen evaluations for gpt, llama and combined
accuracy_llama = accuracy_score(df_llama['m_eval'], df_llama['evaluation_label'])
accuracy_gpt = accuracy_score(df_gpt['m_eval'], df_gpt['evaluation_label'])
combined_accuracy = (accuracy_llama + accuracy_gpt) / 2

print(f"Overall accuracy of 2nd version qwen evaluations for llama: {accuracy_llama:.4f}")
print(f"Overall accuracy of 2nd version qwen evaluations for gpt: {accuracy_gpt:.4f}")
print(f"Combined accuracy: {combined_accuracy:.4f}")

Overall accuracy of 2nd version qwen evaluations for llama: 0.6917
Overall accuracy of 2nd version qwen evaluations for gpt: 0.7417
Combined accuracy: 0.7167


In [21]:
# correct evaluations
correct_llama = df_llama[df_llama['m_eval'] == df_llama['evaluation_label']]
correct_gpt = df_gpt[df_gpt['m_eval'] == df_gpt['evaluation_label']]
correct_overall = pd.concat([correct_llama, correct_gpt])

print(f"Number of correct evaluations for llama: {len(correct_llama)} out of {len(df_llama)}")
print(f"Number of correct evaluations for gpt: {len(correct_gpt)} out of {len(df_gpt)}")
print(f"Number of correct evaluations for combined: {len(correct_overall)} out of {len(pd.concat([df_llama, df_gpt]))}")

Number of correct evaluations for llama: 83 out of 120
Number of correct evaluations for gpt: 89 out of 120
Number of correct evaluations for combined: 172 out of 240


In [22]:
# confusion matrices
conf_matrix_llama = pd.crosstab(df_llama['m_eval'], df_llama['evaluation_label'], rownames=["Manual"],colnames=["Model"])
conf_matrix_gpt = pd.crosstab(df_gpt['m_eval'], df_gpt['evaluation_label'], rownames=["Manual"],colnames=["Model"])

print("Confusion matrix for llama:")
print(conf_matrix_llama)
print("\nConfusion matrix for gpt:")
print(conf_matrix_gpt)

Confusion matrix for llama:
Model   0   1  2   3
Manual              
0       8   5  1   0
1       0  17  9   8
2       1   0  2   1
3       9   2  1  56

Confusion matrix for gpt:
Model   0   1  2   3
Manual              
0       1   0  0   0
1       3  42  6  12
2       1   0  2   2
3       5   2  0  44


In [23]:
# per class accuracies
per_class_llama = df_llama.groupby('m_eval').apply(lambda x: (x["m_eval"] == x["evaluation_label"]).mean())
per_class_gpt = df_gpt.groupby('m_eval').apply(lambda x: (x["m_eval"] == x["evaluation_label"]).mean())

print("\nPer class accuracies for llama:")
print(per_class_llama)
print("\nPer class accuracies for gpt:")
print(per_class_gpt)


Per class accuracies for llama:
m_eval
0    0.571429
1    0.500000
2    0.500000
3    0.823529
dtype: float64

Per class accuracies for gpt:
m_eval
0    1.000000
1    0.666667
2    0.400000
3    0.862745
dtype: float64
